# Imported plow processing notebook

This is the older plow notebook from `GITHUB_NEWER`. It is kept for reference.


In [ ]:
import requests
import os

dataset_url = "https://data.cityofnewyork.us/resource/rmhc-afj9.csv"
rows_per_chunk = 10_000_000
start_chunk = 7   # I was processing 7, 15, 21 because I had to redownload them. Set this to 0 and then set for i in range(40)
#to get all chunks (40 * 100_000_000 rows)
for i in {7,15,21}:  
    offset = i * rows_per_chunk
    params = {
        "$limit": rows_per_chunk,
        "$offset": offset
    }
    print(f"Downloading chunk {i} with offset {offset}...")
    response = requests.get(dataset_url, params=params)

    if response.status_code != 200:
        print(f"Stopped at chunk {i}, HTTP error {response.status_code}")
        break

    content = response.content
    if len(content) < 500:  # heuristic: empty CSV, just header
        print(f"No more rows at chunk {i}. Done.")
        break

    filename = f"plow_{i}.csv"
    with open(filename, "wb") as f:
        f.write(content)

    print(f"Saved {filename}")


Saved plow_7.csv
Saved plow_21.csv
Saved plow_15.csv


In [ ]:
import pandas as pd

years_all = []

for i in {7,15,21}:
    df = pd.read_csv(f"plow_{i}.csv", usecols=["snapshot"], on_bad_lines = "skip")
    df["snapshot"] = pd.to_datetime(df["snapshot"], errors="coerce")
    years = df["snapshot"].dt.year.dropna().astype(int)
    years_all.extend(years)

# Convert to a Series and get value counts (how many rows per year)
year_counts = pd.Series(years_all).value_counts().sort_index()

print(year_counts)

In [10]:
# identifying and dealing with problem in plow_7 and plow_21

import pandas as pd

years_all = []

for i in {7,15,21}:
    df = pd.read_csv(f"plow_{i}.csv", usecols=["snapshot"], on_bad_lines = "skip")
    df["snapshot"] = pd.to_datetime(df["snapshot"], errors="coerce")
    years = df["snapshot"].dt.year.dropna().astype(int)
    years_all.extend(years)
    print(i, df.shape[0])

# Convert to a Series and get value counts (how many rows per year)
year_counts = pd.Series(years_all).value_counts().sort_index()

print(year_counts)

7 10000000
21 10000000
15 10000000
2017    9999994
2018    9999746
2019     715282
2020    9284363
2021        615
Name: count, dtype: int64


In [3]:
with open("plow_21.csv", "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()[-10:]
print("".join(lines))

"71576","2018-09-13T14:54:00.000","2018-09-14T03:37:11.000"
"25440","2018-09-13T14:21:00.000","2018-09-14T03:37:11.000"
"25258","2018-09-13T13:43:00.000","2018-09-14T03:37:11.000"
"19952","2018-09-13T16:19:00.000","2018-09-14T03:37:11.000"
"25252","2018-09-13T15:01:00.000","2018-09-14T03:37:11.000"
"19953","2018-09-13T20:25:00.000","2018-09-14T03:37:11.000"
"23238","2018-09-14T01:04:00.000","2018-09-14T03:37:11.000"
"25253","2018-09-13T15:06:00.000","2018-09-14T03:37:11.000"
"25439","2018-09-13T14:21:00.000","2018-09-14T03:37:11.000"
"8223


In [4]:
with open("plow_21.csv", "rb") as f:
    data = f.read().rsplit(b"\n", 1)[0]  # drop last line

with open("plow_21.csv", "wb") as f:
    f.write(data)

In [ ]:
# issues have been fixed (7, 15, 21 were not read in properly the first time around. checked now that each one has 10 million rows)

In [16]:
# now find range of years

import pandas as pd

years_all = []

for i in range(10):
    df = pd.read_csv(f"plow_{i}.csv", usecols=["snapshot"], on_bad_lines="skip")
    df["snapshot"] = pd.to_datetime(df["snapshot"], errors="coerce")
    years_all.extend(df["snapshot"].dt.year.dropna().astype(int))

# Count occurrences
year_counts = pd.Series(years_all).value_counts().sort_index().to_dict()

year_counts[2023] = 0

print(year_counts)

{2016: 17, 2017: 29792311, 2018: 29999262, 2019: 9999597, 2020: 583107, 2021: 15926118, 2022: 13491591, 2024: 331, 2025: 207666, 2023: 0}


In [17]:
years_next_list = []

for i in range(10, 20):
    df = pd.read_csv(f"plow_{i}.csv", usecols=["snapshot"], on_bad_lines="skip")
    df["snapshot"] = pd.to_datetime(df["snapshot"], errors="coerce")
    years = df["snapshot"].dt.year.dropna().astype(int)
    years_next_list.extend(years)

# Convert to Series and count occurrences
year_counts_next = pd.Series(years_next_list).value_counts()

# Merge with previous year_counts dict
for year, count in year_counts_next.items():
    year_counts[year] = year_counts.get(year, 0) + count

# Sort by year
year_counts = dict(sorted(year_counts.items()))

print(year_counts)

{2016: 17, 2017: 49787232, 2018: 69998368, 2019: 19999013, 2020: 584613, 2021: 30138746, 2022: 22842744, 2023: 0, 2024: 6436550, 2025: 212717}


In [18]:
years_next_list = []

for i in range(20, 30):
    df = pd.read_csv(f"plow_{i}.csv", usecols=["snapshot"], on_bad_lines="skip")
    df["snapshot"] = pd.to_datetime(df["snapshot"], errors="coerce")
    years = df["snapshot"].dt.year.dropna().astype(int)
    years_next_list.extend(years)

# Convert to Series and count occurrences
year_counts_next = pd.Series(years_next_list).value_counts()

# Merge with previous year_counts dict
for year, count in year_counts_next.items():
    year_counts[year] = year_counts.get(year, 0) + count

# Sort by year
year_counts = dict(sorted(year_counts.items()))

print(year_counts)

{2016: 12028779, 2017: 67758470, 2018: 89997905, 2019: 23722860, 2020: 16860051, 2021: 43336134, 2022: 34740870, 2023: 0, 2024: 11342213, 2025: 212718}


In [19]:
years_next_list = []

for i in range(30, 40):
    df = pd.read_csv(f"plow_{i}.csv", usecols=["snapshot"], on_bad_lines="skip")
    df["snapshot"] = pd.to_datetime(df["snapshot"], errors="coerce")
    years = df["snapshot"].dt.year.dropna().astype(int)
    years_next_list.extend(years)

# Convert to Series and count occurrences
year_counts_next = pd.Series(years_next_list).value_counts()

# Merge with previous year_counts dict
for year, count in year_counts_next.items():
    year_counts[year] = year_counts.get(year, 0) + count

# Sort by year
year_counts = dict(sorted(year_counts.items()))

print(year_counts)

{2016: 12028779, 2017: 99209984, 2018: 140075700, 2019: 33721995, 2020: 16861650, 2021: 43336134, 2022: 34740870, 2023: 0, 2024: 11342213, 2025: 212718}


In [20]:
sum(year_counts.values())

391530043

In [8]:
import pandas as pd

plow_1 = pd.read_csv("plow_1.csv")



In [9]:
plow_1[plow_1["physical_id"] == 57761]

,physical_id,last_visited,snapshot
125,57761,2019-02-20T12:35:00.000,2019-02-20T13:07:06.000
109707,57761,2019-02-20T12:35:00.000,2019-02-20T13:22:07.000
219360,57761,2019-02-20T12:35:00.000,2019-02-20T13:37:08.000
329011,57761,2019-02-20T12:35:00.000,2019-02-20T13:52:07.000
438548,57761,2019-02-20T12:35:00.000,2019-02-20T14:07:08.000
...,...,...,...
9534379,57761,2019-03-02T09:17:00.000,2019-03-02T09:38:09.000
9643968,57761,2019-03-02T09:17:00.000,2019-03-02T09:53:33.000
9753557,57761,2019-03-02T09:17:00.000,2019-03-02T10:08:16.000
9863136,57761,2019-03-02T09:17:00.000,2019-03-02T10:23:30.000


In [12]:
cscl = pd.read_csv("CSCL.csv")

In [20]:
cscl[cscl["PHYSICALID"] == 8]

,the_geom,PHYSICALID,L_LOW_HN,L_HIGH_HN,R_LOW_HN,R_HIGH_HN,L_ZIP,R_ZIP,STATUS,BIKE_LANE,...,Post Directional,Post Modifier,Full Street Name,BIKE TRAFFIC DIRECTION,SHAPE__Length,GlobalID,SEGMENT_TYPE,SEGMENT_TYPE_VALUE,STREET NAME,Street Name Label
103418,MULTILINESTRING ((-74.016939165599 40.70482222...,8,NaN,NaN,NaN,NaN,10280.0,10280.0,2,NaN,...,NaN,NaN,BATTERY PL,NaN,12.879389,1a651be8-a549-46da-b78d-decd78b20df0,NaN,NaN,BATTERY,BATTERY PL


In [ ]:
MULTILINESTRING ((-74.22950020729 40.504598328079, -74.229288593263 40.504007445108, -74.230551908419 40.503735199315, -74.23077122522 40.504359026576))

In [17]:
from shapely import wkt

geom = wkt.loads("MULTILINESTRING ((-74.003990902922 40.633997793231, -74.004550255018 40.633422475922))")

coords = []
for line in geom.geoms:
    coords.extend(list(line.coords))

In [18]:
coords

[(-74.003990902922, 40.633997793231), (-74.004550255018, 40.633422475922)]

In [13]:
cscl.shape

(122019, 64)

In [5]:
import pandas as pd
from datetime import datetime

def iter_diffs(csv_path):
    last_seen = {}  # key = (physical_id, day), value = last timestamp

    for chunk in pd.read_csv(csv_path, usecols=['physical_id', 'snapshot'], chunksize=500_000):
        for pid, snap in zip(chunk['physical_id'], chunk['snapshot']):
            try:
                t = datetime.strptime(snap, "%Y-%m-%dT%H:%M:%S.%f")
            except ValueError:
                continue
            day = t.date()
            key = (pid, day)
            if key in last_seen:
                diff_min = (t - last_seen[key]).total_seconds() / 60
                if diff_min < 13 or diff_min > 17:  # outside 13–17 min
                    print(f"physical_id: {pid}, day: {day}, prev: {last_seen[key]}, curr: {t}, diff_min: {diff_min:.2f}")
                yield diff_min
            last_seen[key] = t




In [ ]:
csv_files = [f"plow_{i}.csv" for i in {1}]  # all files
diffs = []

for filename in csv_files:
    print(f"Processing {filename}...")
    diffs.extend(iter_diffs(filename))

s = pd.Series(diffs)
s = s[s < 60]  # filter out gaps >= 1 hour

summary = {
    "avg_diff_min": s.mean(),
    "median_diff_min": s.median(),
    "std_diff_min": s.std(),
    "n_diffs": len(s),
    "prop_near_15": ((s >= 13) & (s <= 17)).mean()
}

print(summary)


In [10]:
# get unique ids

import pandas as pd

unique_ids = set()

# Iterate over all files
for i in range(40):  # adjust to your number of CSVs
    filename = f"plow_{i}.csv"
    print(f"Processing {filename}...")

    # Read in chunks to handle large files
    for chunk in pd.read_csv(filename, usecols=['physical_id'], chunksize=500_000):
        unique_ids.update(chunk['physical_id'].unique())

print(f"Total unique physical_id: {len(unique_ids)}")


Processing plow_0.csv...
Processing plow_1.csv...
Processing plow_2.csv...
Processing plow_3.csv...
Processing plow_4.csv...
Processing plow_5.csv...
Processing plow_6.csv...
Processing plow_7.csv...
Processing plow_8.csv...
Processing plow_9.csv...
Processing plow_10.csv...
Processing plow_11.csv...
Processing plow_12.csv...
Processing plow_13.csv...
Processing plow_14.csv...
Processing plow_15.csv...
Processing plow_16.csv...
Processing plow_17.csv...
Processing plow_18.csv...
Processing plow_19.csv...
Processing plow_20.csv...
Processing plow_21.csv...
Processing plow_22.csv...
Processing plow_23.csv...
Processing plow_24.csv...
Processing plow_25.csv...
Processing plow_26.csv...
Processing plow_27.csv...
Processing plow_28.csv...
Processing plow_29.csv...
Processing plow_30.csv...
Processing plow_31.csv...
Processing plow_32.csv...
Processing plow_33.csv...
Processing plow_34.csv...
Processing plow_35.csv...
Processing plow_36.csv...
Processing plow_37.csv...
Processing plow_38.csv

In [15]:
# Unique PHYSICALID values from CSCL
cscl_ids = set(cscl['PHYSICALID'].unique())

# Find which CSCL IDs never appeared in the Plow dataset
never_plowed = cscl_ids - unique_ids

print(f"Number of streets never plowed: {len(never_plowed)}")
print("Streets never plowed:", [int(x) for x in list(never_plowed)])


Number of streets never plowed: 10862
Streets never plowed: [98306, 196613, 131093, 98327, 30, 131115, 131121, 131122, 196683, 131150, 65643, 196718, 131191, 196729, 196744, 196754, 196759, 196760, 168, 164044, 196817, 196827, 131304, 131305, 164086, 131325, 131330, 131332, 131334, 131335, 131336, 131337, 131338, 131341, 131342, 131344, 196914, 196919, 131388, 131390, 131392, 131407, 131440, 131441, 131447, 131452, 196988, 196998, 98701, 131510, 131511, 131513, 131519, 131520, 131524, 131525, 197060, 131529, 131533, 131534, 197101, 197102, 131569, 197106, 197107, 197108, 197109, 197110, 197111, 197112, 197113, 197115, 197116, 197117, 131584, 197147, 197167, 197168, 164409, 131643, 164411, 131653, 131654, 197191, 584, 131656, 131664, 131665, 131672, 131674, 131681, 131683, 164474, 131709, 131711, 131712, 131713, 131715, 131717, 131718, 131735, 197274, 131742, 131746, 131749, 131752, 131753, 197290, 197293, 66265, 66266, 131801, 131804, 131805, 131808, 131810, 131812, 131814, 164583, 131

In [24]:
cscl[cscl["PHYSICALID"] == 68210]

,the_geom,PHYSICALID,L_LOW_HN,L_HIGH_HN,R_LOW_HN,R_HIGH_HN,L_ZIP,R_ZIP,STATUS,BIKE_LANE,...,Post Directional,Post Modifier,Full Street Name,BIKE TRAFFIC DIRECTION,SHAPE__Length,GlobalID,SEGMENT_TYPE,SEGMENT_TYPE_VALUE,STREET NAME,Street Name Label
70846,MULTILINESTRING ((-73.764917579596 40.75066603...,68210,NaN,NaN,NaN,NaN,11364.0,11364.0,2,NaN,...,NaN,NaN,214 ST,NaN,15.184813,6dbaf38b-27d4-4018-b49c-22578f6c0c8c,NaN,NaN,214,214 ST


In [25]:
cscl[["Borough Code", "Full Street Name"]].head(1000)

,Borough Code,Full Street Name
0,3,AVE N
1,2,HONE AVE
2,4,48 ST
3,1,LAIGHT ST
4,1,W 60 ST
...,...,...
995,5,RODERICK AVE
996,4,108 ST
997,3,AVE M
998,4,ROCKAWAY BLVD


In [ ]:
import pandas as pd

def compute_hourly_coverage(csv_path):
    # Read only necessary columns
    df = pd.read_csv(csv_path, usecols=['physical_id', 'last_visited', 'snapshot'])

    # Convert snapshot to datetime
    df['snapshot'] = pd.to_datetime(df['snapshot'], errors='coerce')
    df = df.dropna(subset=['snapshot'])

    # Floor snapshot to the hour
    df['snap_hour'] = df['snapshot'].dt.floor('H')

    # Group by physical_id and snap_hour
    grouped = df.groupby(['physical_id', 'snap_hour'], observed=True)

    # Compute unique last_visited / total snapshots
    result = grouped.agg(
        unique_visits=('last_visited', 'nunique'),
        n_snapshots=('snapshot', 'count')
    ).reset_index()

    # Compute coverage, handling divide-by-zero
    result['coverage'] = result.apply(
        lambda r: 0 if r['n_snapshots'] == 0 else r['unique_visits'] / r['n_snapshots'],
        axis=1
    )

    # Drop intermediate columns if not needed
    result = result[['physical_id', 'snap_hour', 'coverage']]
    return result


# Process all plow_i files individually
for i in range(10,40):
    path = f'plow_{i}.csv'
    print(f'Processing {path}...')
    hourly_cov = compute_hourly_coverage(path)
    hourly_cov.to_csv(f'plow_coverage_{i}.csv', index=False)
